# C10 — Représentations des encodeurs gelés de la famille B (Brique 4, R6, R7)

Ce notebook passe chaque extrait d'ESC-50, de CREMA-D et de RAVDESS dans trois encodeurs
gelés (wav2vec 2.0 base, mHuBERT-147, encodeur de Whisper base) et garde, pour chaque
extrait, la moyenne dans le temps de chaque couche : un tableau (couches, dimension). Ces
tableaux sont calculés **une seule fois** (R6) et publiés en datasets Kaggle privés : les
têtes de C11 s'entraînent ensuite dessus en quelques secondes, sans repasser par les
encodeurs.

Tout le calcul vient du dépôt (`echolens.training.encoders` et
`echolens.training.embeddings`) ; le notebook enchaîne les étapes et vérifie les sorties.
Aucun fichier de représentations ne va sur GitHub.

## a. Démarrage

On clone le dépôt (branche `BRANCH`) et on l'installe en mode éditable (I9), **sans** l'extra
`[ml]` : torch et transformers sont déjà installés sur Kaggle. L'installation éditable n'est
visible qu'après un redémarrage du noyau : on ajoute donc `src/` au `sys.path`.

In [ ]:
import sys

BRANCH = "main"
REPO_URL = "https://github.com/mentalist011101/TP_SIGNAL_DS.git"
CLONE_DIR = "/kaggle/working/TP_SIGNAL_DS"

!rm -rf {CLONE_DIR}
!git clone --depth 1 --branch {BRANCH} {REPO_URL} {CLONE_DIR}
# Kaggle tourne sous Python 3.13, au-delà de requires-python (<3.13) du dépôt.
%pip install -q --ignore-requires-python -e {CLONE_DIR}

sys.path.insert(0, CLONE_DIR + "/src")
print(sys.version)

## b. ffmpeg et GPU

`load_audio` décode avec ffmpeg (D1). Le calcul se fait sur le GPU (T4), choisi
explicitement : sans GPU, on arrête plutôt que de calculer pendant des heures sur CPU.

In [ ]:
import shutil

import torch
import transformers

if shutil.which("ffmpeg") is None:
    !apt-get -qq update && apt-get -qq install -y ffmpeg
print("ffmpeg :", shutil.which("ffmpeg"))

if not torch.cuda.is_available():
    raise RuntimeError("Aucun GPU : activer l'accélérateur GPU T4 du notebook.")
device = torch.device("cuda", 0)
print("GPU :", torch.cuda.get_device_name(device))
print("torch", torch.__version__, "· transformers", transformers.__version__)

## c. Trouver les jeux

Les manifestes donnent des chemins relatifs à `<datasets_dir>/<id du jeu>` (docs/datasets.md).
On cherche les copies attachées au kernel, comme C3 et C5 :

- CREMA-D : un dossier `AudioWAV` qui contient exactement 7 442 WAV (copie `ejlok1/cremad`) ;
- RAVDESS : un dossier qui contient `Actor_01` à `Actor_24`, chaque fichier une seule fois
  (copie `uwrfkaggler/ravdess-emotional-speech-audio`) ;
- ESC-50 : `meta/esc50.csv`, sinon l'archive officielle téléchargée depuis GitHub.

Puis on crée `/kaggle/working/datasets/esc50`, `cremad` et `ravdess`, liens symboliques vers
les racines trouvées.

In [ ]:
import urllib.request
import zipfile
from pathlib import Path

INPUT_DIR = Path("/kaggle/input")
DATASETS_DIR = Path("/kaggle/working/datasets")
DOWNLOAD_DIR = Path("/kaggle/working/downloads")
ESC50_ARCHIVE_URL = "https://github.com/karolpiczak/ESC-50/archive/refs/heads/master.zip"
CREMAD_EXPECTED_WAV = 7442
RAVDESS_EXPECTED_WAV = 1440


def find_cremad_root() -> Path:
    # Racine = dossier parent d'un AudioWAV qui contient exactement 7 442 WAV.
    for audio_folder in sorted(INPUT_DIR.glob("**/AudioWAV")):
        wav_count = len(list(audio_folder.glob("*.wav")))
        if wav_count == CREMAD_EXPECTED_WAV:
            return audio_folder.parent
    raise RuntimeError("CREMA-D introuvable : attacher le dataset ejlok1/cremad.")


def find_ravdess_root() -> Path:
    # Racine = dossier qui contient Actor_01 à Actor_24, chaque nom de fichier une seule fois.
    candidates = sorted({actor.parent for actor in INPUT_DIR.glob("**/Actor_01")})
    for candidate in candidates:
        wav_paths = list(candidate.glob("Actor_*/*.wav"))
        unique_names = {path.name for path in wav_paths}
        if len(wav_paths) == RAVDESS_EXPECTED_WAV and len(unique_names) == RAVDESS_EXPECTED_WAV:
            return candidate
    raise RuntimeError("RAVDESS introuvable : attacher uwrfkaggler/ravdess-emotional-speech-audio.")


def find_esc50_root() -> Path:
    found = sorted(INPUT_DIR.glob("**/meta/esc50.csv"))
    if found:
        return found[0].parent.parent
    # Archive officielle (~600 Mo), extraite dans /kaggle/working/downloads.
    DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
    archive_path = DOWNLOAD_DIR / "esc50.zip"
    urllib.request.urlretrieve(ESC50_ARCHIVE_URL, archive_path)
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall(DOWNLOAD_DIR)
    archive_path.unlink()
    return DOWNLOAD_DIR / "ESC-50-master"


roots = {
    "esc50": find_esc50_root(),
    "cremad": find_cremad_root(),
    "ravdess": find_ravdess_root(),
}
DATASETS_DIR.mkdir(parents=True, exist_ok=True)
for dataset_id, root in roots.items():
    link = DATASETS_DIR / dataset_id
    if link.is_symlink():
        link.unlink()
    link.symlink_to(root)
    print(dataset_id, "→", root)

## d. Manifestes versionnés

On lit les manifestes de `data/manifests/` (construits par C3 et C5) : l'ordre de leurs lignes
est celui des fichiers de représentations. On vérifie d'abord que chaque audio existe.

In [ ]:
from echolens.core.data.manifest import audio_path, load_manifest
from echolens.training.embeddings import load_embeddings_config

embeddings_config = load_embeddings_config()
manifests_dir = Path(CLONE_DIR) / "data" / "manifests"
manifests = {}
for dataset_id in embeddings_config.datasets:
    rows = load_manifest(manifests_dir / f"{dataset_id}.csv")
    missing = [row.path for row in rows if not audio_path(row, DATASETS_DIR).exists()]
    if missing:
        raise RuntimeError(f"{dataset_id} : {len(missing)} audios introuvables, ex. {missing[:3]}")
    manifests[dataset_id] = rows
    print(dataset_id, ":", len(rows), "lignes, tous les audios sont présents")

## e. Calcul des représentations

Pour chaque encodeur, puis chaque jeu : un extrait à la fois, sans remplissage ni masque
d'attention, `model.eval()` et `torch.no_grad()`, en float32. Pour Whisper, la même passe
donne deux moyennes : sur les trames utiles (la version utilisée) et sur les 1 500 trames
(dossier `padded/`, seulement pour mesurer l'effet du remplissage).

Sortie : `/kaggle/working/embeddings/<encodeur>/<jeu>/` avec `embeddings.npy`
(extraits, couches, dimension), `index.csv` (dataset, path) et `metadata.json`.

In [ ]:
import gc

from echolens.training.embeddings import manifest_index, save_embeddings
from echolens.training.encoders import embed_manifest, load_encoder, make_metadata

OUTPUT_DIR = Path("/kaggle/working/embeddings")

for encoder_config in embeddings_config.encoders:
    encoder = load_encoder(encoder_config, embeddings_config.sample_rate, device)
    for dataset_id, rows in manifests.items():
        result = embed_manifest(rows, DATASETS_DIR, encoder)
        index = manifest_index(rows)
        metadata = make_metadata(encoder, dataset_id, result, padded=False)
        output_dir = OUTPUT_DIR / encoder_config.id / dataset_id
        save_embeddings(output_dir, result.embeddings, index, metadata)
        if result.padded is not None:
            padded_metadata = make_metadata(encoder, dataset_id, result, padded=True)
            padded_dir = OUTPUT_DIR / encoder_config.id / "padded" / dataset_id
            save_embeddings(padded_dir, result.padded, index, padded_metadata)
        seconds = round(result.compute_seconds)
        print(f"{encoder_config.id} / {dataset_id} : {result.embeddings.shape}, {seconds} s")
    # Libère le GPU avant l'encodeur suivant.
    del encoder
    gc.collect()
    torch.cuda.empty_cache()

## f. Contrôles

On relit chaque dossier : autant de lignes que dans le manifeste, même ordre (`index.csv`),
aucune valeur manquante (NaN) ou infinie. Sinon, on arrête le notebook.

In [ ]:
import numpy as np
import pandas as pd

from echolens.training.embeddings import load_embeddings

summary_rows = []
for folder in sorted(OUTPUT_DIR.glob("**/embeddings.npy")):
    files = load_embeddings(folder.parent)
    metadata = files.metadata
    expected_index = manifest_index(manifests[metadata.dataset])
    if files.index != expected_index:
        raise RuntimeError(f"{folder.parent} : index différent du manifeste")
    row_count = len(files.embeddings)
    if row_count != len(expected_index):
        raise RuntimeError(f"{folder.parent} : {row_count} lignes au lieu de {len(expected_index)}")
    if not np.all(np.isfinite(files.embeddings)):
        raise RuntimeError(f"{folder.parent} : valeurs NaN ou infinies")
    summary_rows.append(
        {
            "encodeur": metadata.encoder_id,
            "jeu": metadata.dataset,
            "remplissage": metadata.padded,
            "forme": str(files.embeddings.shape),
            "Mo": round(folder.stat().st_size / 1e6, 1),
            "secondes": round(metadata.compute_seconds),
            "do_normalize": metadata.do_normalize,
            "GPU": metadata.gpu,
        }
    )
summary = pd.DataFrame(summary_rows)
display(summary)
summary.to_csv("/kaggle/working/c10_summary.csv", index=False)

## g. Diagnostic du remplissage de Whisper

Sur ESC-50 et CREMA-D : part moyenne des trames utiles, puis similarité cosinus moyenne entre
200 paires d'extraits tirées au hasard (graine 42), sur la dernière couche de Whisper, avec et
sans remplissage. Si la moyenne inclut le silence de remplissage, tous les extraits se
ressemblent : la similarité monte vers 1 et la représentation discrimine moins.

In [ ]:
import json

from echolens.training.embeddings import mean_cosine_of_random_pairs

DIAGNOSTIC_DATASETS = ["esc50", "cremad"]
PAIR_COUNT = 200
SEED = 42
DIAGNOSTIC_PATH = Path("/kaggle/working/c10_whisper_padding_diagnostic.json")

diagnostic = {}
for dataset_id in DIAGNOSTIC_DATASETS:
    useful = load_embeddings(OUTPUT_DIR / "whisper-base" / dataset_id)
    padded = load_embeddings(OUTPUT_DIR / "whisper-base" / "padded" / dataset_id)
    # Dernière couche de l'encodeur : index -1 sur l'axe des couches.
    useful_last = useful.embeddings[:, -1, :]
    padded_last = padded.embeddings[:, -1, :]
    diagnostic[dataset_id] = {
        "mean_useful_frame_share": useful.metadata.mean_useful_frame_share,
        "cosine_useful_frames": mean_cosine_of_random_pairs(useful_last, PAIR_COUNT, SEED),
        "cosine_padded": mean_cosine_of_random_pairs(padded_last, PAIR_COUNT, SEED),
    }
print(json.dumps(diagnostic, indent=2))
with DIAGNOSTIC_PATH.open("w", encoding="utf-8") as handle:
    json.dump(diagnostic, handle, indent=2)

## Nettoyage

On supprime les liens vers les jeux, l'archive d'ESC-50 et le clone du dépôt : les sorties du
notebook ne gardent que les représentations, le résumé et le diagnostic.

In [ ]:
shutil.rmtree(DATASETS_DIR, ignore_errors=True)
shutil.rmtree(DOWNLOAD_DIR, ignore_errors=True)
shutil.rmtree(CLONE_DIR, ignore_errors=True)
print(sorted(path.name for path in Path("/kaggle/working").iterdir()))